1) r"..."

In [2]:
import re

# 나쁨: \n 이 줄바꿈으로 먼저 처리됨
p1 = "\bword\b"      # 의도와 다름
print('p1:', p1)

# 좋음: 정규식 엔진에 \b 그대로 전달
p2 = r"\bword\b"

print('p2:', p2)  # \bword\b


p1: word
p2: \bword\b


## 2) 핵심 문법(cheatsheet)
* 문자 클래스(character class): [abc], [^0-9]
* 축약 클래스(shorthand class): \d(숫자), \w(단어문자), \s(공백), 대문자는 반대 의미 \D, \W, \S
* 수량자(quantifier): * (0+), + (1+), ? (0 또는 1), {m,n}
* 탐욕적/게으름(Greedy/Lazy): *, +, ?는 기본 탐욕적, 뒤에 ?를 붙이면 게으름 (예: .*?)
* 앵커(anchor): ^(문자열/라인 시작), $(끝), \A(문자열 시작), \Z(문자열 끝), \b(단어 경계), \B(비-경계)
* 그룹(group): ( ... ) 캡처, (?: ... ) 비캡처(non-capturing), (?P<name>...) 이름있는 그룹
* 대안(alternation): A|B
* 전방/후방 탐색(lookaround): (?=...) 긍정 전방, (?!...) 부정 전방, (?<=...) 긍정 후방, (?<!...) 부정 후방, (?:...) 소비 전방
* 플래그(flags): re.IGNORECASE(i), re.MULTILINE(m), re.DOTALL(s), re.VERBOSE(x), re.ASCII(a), re.UNICODE(u, 기본)

## 3) re 기본 함수 사용법

In [10]:
import re

text = "Contact: alice@example.com, bob@test.org"

# 1) search: 처음 매치 1개
m = re.search(r"\w+@\w+\.\w+", text)
print("1. search:", m.group())   # m.group()은 m.group(0)과 동일함. 매치된 전체 문자열을 저장함

# 2) findall: 전부 추출(문자열 리스트)
all_emails = re.findall(r"\w+@\w+\.\w+", text)
print("2. findall:", all_emails)

# 3) finditer: 전부 반복자(매치 객체) — 위치 정보 등 활용
for m in re.finditer(r"\w+@\w+\.\w+", text):
    print("3. finditer:", m.group(), m.span())  # alice@example.com (9, 26), bob@test.org (28, 40)

# 4) match: 문자열 **시작**에서만 매치
print("4. match:", re.match(r"Contact", text) is not None)

# 5) sub: 치환(replace)
masked = re.sub(r"(\w)@\w+", r"\1@****", text)   # "\1"은 첫번째 캡쳐 그룹 참조
print("5. sub:", masked)

# 6) split: 구분자 기준 분리(정규식 사용)
parts = re.split(r"[,\s]+", text)
print("6. split:", parts)

# 7) compile: 패턴 재사용(성능) — 고빈도 호출 시 권장
email_pat = re.compile(r"\w+@\w+\.\w+")
print("7. compiled findall:", email_pat.findall(text))


1. search: alice@example.com
2. findall: ['alice@example.com', 'bob@test.org']
3. finditer: alice@example.com (9, 26)
3. finditer: bob@test.org (28, 40)
4. match: True
5. sub: Contact: alice@****.com, bob@****.org
6. split: ['Contact:', 'alice@example.com', 'bob@test.org']
7. compiled findall: ['alice@example.com', 'bob@test.org']


## 4) 문자 클래스/수량자/탐욕 vs 게으름

In [13]:
import re

html = "<p>first</p><p>second</p>"

# 탐욕적(Greedy): 가능한 많이, 넓게 검색
print(re.findall(r"<p>.*</p>", html))
# => ['<p>first</p><p>second</p>']  (전부 먹어버림)

# 게으른(Lazy): 가능한 적게, 좁게 검색
print(re.findall(r"<p>.*?</p>", html))
# => ['<p>first</p>', '<p>second</p>']  (원하는 덩어리별 추출)


['<p>first</p><p>second</p>']
['<p>first</p>', '<p>second</p>']


## 5) 앵커와 플래그의 조합

In [14]:
import re

text = "first line\nSecond line\nthird LINE"

# MULTILINE: ^, $가 각 줄의 시작/끝으로 동작
print(re.findall(r"^[a-z]+\sline$", text, flags=re.MULTILINE))
# => ['first line', 'third LINE']  (대소문자 무시 안 함)

# IGNORECASE 추가
print(re.findall(r"^[a-z]+\sline$", text, flags=re.MULTILINE|re.IGNORECASE))
# => ['first line', 'Second line', 'third LINE']

# DOTALL: . 이 줄바꿈까지 포함
print(re.findall(r"first.*LINE", text, flags=re.DOTALL))
# => ['first line\nSecond line\nthird LINE']


['first line']
['first line', 'Second line', 'third LINE']
['first line\nSecond line\nthird LINE']


## 6) 그룹과 참조(Backreference)

In [20]:
import re

s = "2025-09-04, 2025/09/04, 2025.09.04"

# 같은 구분자만 허용(하나의 그룹을 뒤에서 \1로 재사용)
pat = re.compile(r"(\d{4})([-/.])(\d{2})\2(\d{2})")   # "\2" 는 두번째 캐쳐그룹([-/.])을 의미함
for m in pat.finditer(s):
    print(m.group(), "Y:", m.group(1), "sep:", m.group(2) )

# 이름 있는 그룹
pat2 = re.compile(r"(?P<y>\d{4})(?P<sep>[-/.])(?P<m>\d{2})(?P=sep)(?P<d>\d{2})")
m = pat2.search("today 2025-09-04")
print(m.group("y"), m.group("m"), m.group("d"))


2025-09-04 Y: 2025 sep: -
2025/09/04 Y: 2025 sep: /
2025.09.04 Y: 2025 sep: .
2025 09 04


## 7) 전방/후방 탐색(Lookaround) — (미)소비 없는 조건

In [21]:
import re

text = "price: $99, $199, 199 won, $5!"

# 달러 기호가 **앞에** 있는 숫자만 (긍정 전방 (?=\d))
print(re.findall(r"\$(?=\d)\d+", text))  # ['$99', '$199', '$5']

# 달러 없는 숫자만 (부정 전방 (?!\$))
print(re.findall(r"(?<!\$)\b\d+\b", text))  # ['199']   \b는 단어 경계(boundary)

# 'http' 뒤에 's'가 오는 경우만 (긍정 전방)
urls = ["http://a.com", "https://b.com"]
print([u for u in urls if re.search(r"^http(?=s)://", u)])
# => ['https://b.com']


['$99', '$199', '$5']
['199']
[]


## 8) sub 고급: 콜백 함수로 치환

In [23]:
import re

text = "Order A-12, B-007, C-3"

def pad_numbers(m):   # 콜백함수
    # 그룹1: 문자코드, 그룹2: 숫자
    code, num = m.group(1), m.group(2)
    return f"{code}-{int(num):03d}"

print(re.sub(r"([A-Z])-([0-9]+)", pad_numbers, text))
# => Order A-012, B-007, C-003


Order A-012, B-007, C-003


## 9) 가독성 향상: re.VERBOSE(x, 확장 모드)

In [24]:
import re
# flags=re.VERBOSE는 공백이나 개행문자 주석 등을 무시하고 여러 행을 처리함
pattern = re.compile(r"""
    (?P<user>[a-zA-Z0-9._%+-]+)   # 사용자
    @
    (?P<host>[a-zA-Z0-9.-]+)      # 호스트
    \.
    (?P<tld>[A-Za-z]{2,})         # 최상위 도메인, 2개 이상
""", flags=re.VERBOSE)

print(pattern.findall("alice@example.com bob@test.org"))
# => [('alice', 'example', 'com'), ('bob', 'test', 'org')]


[('alice', 'example', 'com'), ('bob', 'test', 'org')]


## 10) 한국 실전 예시

In [28]:
# 휴대폰 번호 (단순화 버전)
import re

phones = """
010-1234-5678
01012345678
011-987-6543
02-123-4567
"""

# 휴대폰(010/011/016/017/018/019)만 추출, 구분자 있거나 없이
pat = re.compile(r"\b01[016-9][ -]?\d{3,4}[ -]?\d{4}\b")  # [ -] 공백이나 하이픈, \s는 공백과 탭, 개행, 개리지리턴 등
print(pat.findall(phones))


['010-1234-5678', '01012345678', '011-987-6543']


list

In [29]:
# 주민번호 형태 감지(마스킹 예시) — 실제 검증은 별도 체크섬 필요
import re

data = "주민번호는 901010-1234567 이고, 또다른 표기는 9001011234567 입니다."
masked = re.sub(r"\b(\d{6})[-]?\d{7}\b", r"\1-*******", data)
print(masked)


주민번호는 901010-******* 이고, 또다른 표기는 900101-******* 입니다.


In [32]:
# 날짜 파싱(YYYY-MM-DD, YYYY/MM/DD, YYYY.MM.DD)
import re

text = "마감: 2025-09-04, 이전: 2024/12/31, 로그: 2023.01.01"
pat = re.compile(r"(?P<y>\d{4})[-/.](?P<m>0[1-9]|1[0-2])[-/.](?P<d>0[1-9]|[12]\d|3[01])")
for m in pat.finditer(text):
    print(m.group(), m.groupdict())  # m.group(1) 등도 가능


2025-09-04 {'y': '2025', 'm': '09', 'd': '04'}
2024/12/31 {'y': '2024', 'm': '12', 'd': '31'}
2023.01.01 {'y': '2023', 'm': '01', 'd': '01'}


In [33]:
# Apache/Nginx 엑세스 로그에서 IP, 요청, 상태코드 추출
import re

log = '127.0.0.1 - - [21/Jul/2025:10:21:07 +0900] "GET /index.html HTTP/1.1" 200 532'

pat = re.compile(
    r'(?P<ip>\d{1,3}(?:\.\d{1,3}){3})\s+-\s+-\s+\[(?P<time>[^\]]+)\]\s+'
    r'"(?P<method>GET|POST|PUT|DELETE|HEAD|OPTIONS)\s+(?P<path>\S+)\s+HTTP/(?P<httpver>[^"]+)"\s+'
    r'(?P<status>\d{3})\s+(?P<size>\d+)'
)

m = pat.search(log)
print(m.groupdict())


{'ip': '127.0.0.1', 'time': '21/Jul/2025:10:21:07 +0900', 'method': 'GET', 'path': '/index.html', 'httpver': '1.1', 'status': '200', 'size': '532'}


In [34]:
# CSV 라인 정리: 공백 다중 → 하나
import re
line = "name,   age,   city"
print(re.sub(r"\s+", " ", line))  # "name, age, city"


name, age, city


In [37]:
# 태그 제거(단순 HTML 스트리퍼 — 한계 있음)
import re
html = "<div>Hello <b>world</b> &nbsp;!</div>"

text = re.sub(r"<[^>]+>", "", html)       # 태그 제거
print(text.strip())

text = re.sub(r"&nbsp;?", " ", text)      # HTML 엔티티 일부 치환
print(text.strip())


Hello world &nbsp;!
Hello world  !


In [38]:
# 한국 우편번호(5자리)만 찾기
import re
s = "배송지 06236, 구 우편 123-456, 메모 12345."
print(re.findall(r"\b\d{5}\b", s))
# 기대: ['06236', '12345']


['06236', '12345']


In [39]:
# Markdown 링크 추출: [텍스트](URL)
import re
md = "문서: [구글](https://google.com), [오픈AI](https://openai.com)"
pat = re.compile(r"\[([^\]]+)\]\((https?://[^)]+)\)")
print(pat.findall(md))
# 기대: [('구글', 'https://google.com'), ('오픈AI', 'https://openai.com')]


[('구글', 'https://google.com'), ('오픈AI', 'https://openai.com')]


In [40]:
# 동일 단어가 두 번 연속 나오는지(Backreference)
import re
s = "This is is a test. That that is not."
print(re.findall(r"\b(\w+)\s+\1\b", s, flags=re.IGNORECASE))
# 기대: ['is', 'that']


['is', 'That']


In [41]:
# 숫자 사이에만 있는 콤마 제거(lookaround)
import re
n = "1,234,567원 / code,info"
print(re.sub(r"(?<=\d),(?=\d)", "", n))   # (?<=\d), 긍정후방탐색:"," 뒤에 숫자가 온다
# 기대: "1234567원 / code,info"


1234567원 / code,info


In [42]:
# re.subn(치환 횟수), pattern.fullmatch
import re

print(re.subn(r"\s+", " ", "a   b   c"))  # ('a b c', 2)  # (결과문자열, 치환횟수)

# 문자열 전체가 패턴과 정확히 일치하는지?
print(re.fullmatch(r"[A-Z]{3}\d{2}", "ABC12") is not None)  # True
print(re.fullmatch(r"[A-Z]{3}\d{2}", "xABC12y") is not None)  # False


('a b c', 2)
True
False


## 자동차 번호판 이미지에서 라벨 추출하기
#### 한국 자동차 번호판 이미지로부터 라벨을 추출하려고 하는데 아래의 규칙을 따라야 합니다

* "32가1234_523435_s1.jpg -> 32가1234
* "123라1383_6453345_s1.jpg -> 123라1383
* "전북96사2268_8756345_s1.jpg -> 전북96사2268
* "08루3300_65433456_d1" -> 08루
* "08루3300_65433456_d2" -> 3300
* "서울32가1234_23456663_d1.jpg" -> 서울32
* "서울32가1234_23456663_d2.jpg" -> 가1234
* "서울4퍼3151_8763456_d1.jpg" -> 서울4
* "서울4퍼3151_8763456_d2.jpg" -> 퍼3151

## 위의 규칙에 따라 파이썬 정규표현식을 사용하여 라벨을 추출하는 예제
* 힌트: "s1", "d1", "d2" 식별자에 따라서 추출하는 텍스트가 달라짐

In [43]:
# 자동차 번호판
import re

# 1) 파일명에서 번호판문자열(plate)과 세그먼트(seg: s1/d1/d2) 추출
FNAME_RE = re.compile(
    r"""
    ^(?P<plate>[^_]+)      # 언더스코어 전까지가 '번호판 문자열'
    _\d+                    # 임의의 숫자 구간
    _(?P<seg>s1|d1|d2)      # 세그먼트 식별자
    (?:\.\w+)?$             # 확장자는 있을 수도 없을 수도
    """,
    re.VERBOSE
)

# 2) 번호판 문자열을 구조적으로 파싱
#    [지역(선택, 2글자 이상 한글)] + [숫자 1~3자리] + [한글 1글자] + [숫자 4자리]
PLATE_RE = re.compile(
    r"""
    ^(?P<region>[가-힣]{2,})?  # 지역(예: 서울, 전북 등) - 없을 수 있음
    (?P<num>\d{1,3})           # 숫자 1~3자리 (예: 32, 123, 08 등)
    (?P<char>[가-힣])          # 한글 1글자 (예: 가, 사, 루, 퍼 등)
    (?P<tail>\d{4})$           # 마지막 4자리 숫자
    """,
    re.VERBOSE
)

def extract_label(filename: str) -> str:
    """파일명에서 s1/d1/d2 규칙에 맞는 라벨을 추출"""
    m = FNAME_RE.match(filename)
    if not m:
        raise ValueError(f"파일명 형식 불일치: {filename}")

    plate = m.group("plate")
    seg = m.group("seg")

    if seg == "s1":
        # s1: 전체 번호판
        return plate

    # d1, d2는 번호판 구조 파싱 필요
    p = PLATE_RE.match(plate)
    if not p:
        raise ValueError(f"번호판 구조 파싱 실패: {plate}")

    region = p.group("region") or ""
    num    = p.group("num")
    char   = p.group("char")
    tail   = p.group("tail")

    if seg == "d1":
        # 지역이 있으면 '지역+숫자', 없으면 '숫자+한글'
        return f"{region}{num}" if region else f"{num}{char}"

    if seg == "d2":
        # 지역이 있으면 '한글+끝 4자리', 없으면 '끝 4자리'만
        return f"{char}{tail}" if region else tail

    # 방어적(여기 도달하지 않음)
    raise ValueError(f"미지원 세그먼트: {seg}")

# ------------------ 테스트 ------------------
tests = [
    # s1
    ("32가1234_523435_s1.jpg", "32가1234"),
    ("123라1383_6453345_s1.jpg", "123라1383"),
    ("전북96사2268_8756345_s1.jpg", "전북96사2268"),
    # d1
    ("08루3300_65433456_d1", "08루"),
    ("서울32가1234_23456663_d1.jpg", "서울32"),
    ("서울4퍼3151_8763456_d1.jpg", "서울4"),
    # d2
    ("08루3300_65433456_d2", "3300"),
    ("서울32가1234_23456663_d2.jpg", "가1234"),
    ("서울4퍼3151_8763456_d2.jpg", "퍼3151"),
]

for fn, expected in tests:
    out = extract_label(fn)
    print(f"{fn:35s} -> {out:8s} | ok? {out == expected}")


32가1234_523435_s1.jpg               -> 32가1234  | ok? True
123라1383_6453345_s1.jpg             -> 123라1383 | ok? True
전북96사2268_8756345_s1.jpg            -> 전북96사2268 | ok? True
08루3300_65433456_d1                 -> 08루      | ok? True
서울32가1234_23456663_d1.jpg           -> 서울32     | ok? True
서울4퍼3151_8763456_d1.jpg             -> 서울4      | ok? True
08루3300_65433456_d2                 -> 3300     | ok? True
서울32가1234_23456663_d2.jpg           -> 가1234    | ok? True
서울4퍼3151_8763456_d2.jpg             -> 퍼3151    | ok? True
